# 01 — Generate Synthetic YOLO Instance Segmentation Dataset

Generate a YOLO instance segmentation dataset for **IQ Noodles** pieces using BlenderProc.

**Input:** `IQ-Noodles.stl`

**Output:** `data/noodles_seg_dataset/` with images, labels, dataset.yaml, preview.jpg

**Architecture rule:** This notebook never imports `blenderproc`. All BlenderProc logic lives in `blenderproc_scene_generator.py`, called via subprocess.

## Section 0 — Dependencies & Config

In [ ]:
# ── UNIVERSAL SETUP & CONFIGURATION ───────────────────────────────────────────
# All common imports, constants, and paths used across the entire pipeline.
# No assertions for intermediate files are done here.

import os
import sys
import json
import random
import shutil
import subprocess
import zipfile
import urllib.request
from pathlib import Path
from collections import Counter

import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFilter
import trimesh
import yaml
from ultralytics import YOLO

# ── Project root ──────────────────────────────────────────────────────────────
NOTEBOOK_DIR = Path(os.getcwd())
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
print(f"Project root: {PROJECT_ROOT}")

# ── Main Directories ──────────────────────────────────────────────────────────
DATA_DIR       = PROJECT_ROOT / 'data'
DATASET_DIR    = DATA_DIR / 'noodles_seg_dataset'
TRAINING_DIR   = DATA_DIR / 'noodles_training'
FINETUNE_DIR   = DATA_DIR / 'noodles_finetune_dataset'
MODELS_DIR     = PROJECT_ROOT / 'models' / 'piece_segmentor'

# ── Synthetic Data Configuration ──────────────────────────────────────────────
STL_PATH       = PROJECT_ROOT / 'IQ-Noodles.stl'
BOARD_EXTENTS_MM = [140.0, 140.0]
GENERATOR_SCRIPT = PROJECT_ROOT / 'blenderproc_scene_generator.py'

NUM_TRAIN_IMAGES = 600
NUM_VAL_IMAGES = 100
MIN_PIECES_PER_IMAGE = 3
MAX_PIECES_PER_IMAGE = 11

# ── Piece Definitions ─────────────────────────────────────────────────────────
PIECE_LABELS = list('ABCDEFGHIJK')
NUM_CLASSES  = len(PIECE_LABELS)

# Colors for bounding boxes / drawing (RGB)
PIECE_COLORS = {
    'A': ('YellowGreen', (0x95, 0xD4, 0x50)),
    'B': ('Red',         (0xEE, 0x39, 0x4F)),
    'C': ('DarkBlue',    (0x20, 0x6D, 0xD9)),
    'D': ('Purple',      (0xC7, 0x78, 0xB9)),
    'E': ('Orange',      (0xFC, 0x69, 0x0C)),
    'F': ('SkyBlue',     (0x08, 0xA7, 0xE8)),
    'G': ('Green',       (0x1F, 0xA1, 0x5B)),
    'H': ('DarkRed',     (0xB6, 0x30, 0x48)),
    'I': ('Yellow',      (0xF9, 0xD6, 0x5E)),
    'J': ('Pink',        (0xEC, 0x71, 0xA8)),
    'K': ('Teal',        (0x85, 0xDA, 0xBB)),
}

# Linear RGB for Blender rendering
PIECE_COLORS_LINEAR = {
    "A": (0.322, 0.660, 0.064),
    "B": (0.846, 0.049, 0.080),
    "C": (0.016, 0.165, 0.726),
    "D": (0.576, 0.213, 0.512),
    "E": (0.973, 0.149, 0.003),
    "F": (0.001, 0.431, 0.855),
    "G": (0.014, 0.390, 0.124),
    "H": (0.472, 0.032, 0.073),
    "I": (0.953, 0.690, 0.128),
    "J": (0.851, 0.165, 0.420),
    "K": (0.247, 0.726, 0.519),
}

# Unified class mapping for fine-tuning
UNIFIED_NAMES = {
    i: f"{label}_{PIECE_COLORS[label][0]}"
    for i, label in enumerate(PIECE_LABELS)
}

print(f"Initialized unified setup variables over {NUM_CLASSES} classes.")

## Section 1 — STL Loading & OBJ Export

In [ ]:
# ── Load the STL ──────────────────────────────────────────────────────────────
mesh = trimesh.load(str(STL_PATH))
print(f"Mesh type: {type(mesh).__name__}")

# ── Split into components ─────────────────────────────────────────────────────
if isinstance(mesh, trimesh.Trimesh):
    components = mesh.split(only_watertight=False)
elif isinstance(mesh, trimesh.Scene):
    components = []
    for name, geom in mesh.geometry.items():
        if isinstance(geom, trimesh.Trimesh):
            components.extend(geom.split(only_watertight=False))
else:
    raise ValueError(f"Unexpected mesh type: {type(mesh)}")

print(f"Total components: {len(components)}")

# ── Classify: board / artifact / piece ─────────────────────────────────────────
board_parts = []
piece_parts = []
artifact_count = 0

for comp in components:
    ext = comp.extents
    if ext.max() > 80:
        board_parts.append(comp)
        print(f"  [board] {len(comp.vertices)} verts, extents={np.round(ext, 1)}")
    elif len(comp.faces) < 100 or ext.min() < 1.0:
        artifact_count += 1
        print(f"  [skip]  {len(comp.faces)} faces, extents={np.round(ext, 1)}")
    else:
        piece_parts.append(comp)

print(f"\n  Board parts: {len(board_parts)}, Artifacts: {artifact_count}, Pieces: {len(piece_parts)}")

# ── Combine board ─────────────────────────────────────────────────────────────
assert len(board_parts) > 0, "No board parts found!"
board_mesh = trimesh.util.concatenate(board_parts)
print(f"  Board mesh: {len(board_mesh.vertices)} verts, {len(board_mesh.faces)} faces")

# ── Assert exactly 11 pieces ──────────────────────────────────────────────────
assert len(piece_parts) == 11, f"Expected 11 pieces, got {len(piece_parts)}"

# ── Sort by face count and map to labels ──────────────────────────────────────
piece_parts.sort(key=lambda p: len(p.faces))
label_to_piece = dict(zip(FACE_SORTED_LABELS, piece_parts))
pieces = [label_to_piece[label] for label in PIECE_LABELS]  # reorder to A-K

# ── Save board extents ────────────────────────────────────────────────────────
BOARD_EXTENTS_MM = board_mesh.extents
print(f"  Board extents (mm): {np.round(BOARD_EXTENTS_MM, 1)}")

# ── Export OBJ files ──────────────────────────────────────────────────────────
PIECE_OBJS_DIR.mkdir(parents=True, exist_ok=True)

for i, label in enumerate(PIECE_LABELS):
    obj_path = PIECE_OBJS_DIR / f"piece_{label}.obj"
    pieces[i].export(str(obj_path))

board_obj_path = PIECE_OBJS_DIR / 'board.obj'
board_mesh.export(str(board_obj_path))

# ── Verify all files exist ────────────────────────────────────────────────────
for label in PIECE_LABELS:
    assert (PIECE_OBJS_DIR / f"piece_{label}.obj").exists(), f"Missing piece_{label}.obj"
assert board_obj_path.exists(), "Missing board.obj"

# ── Print summary ─────────────────────────────────────────────────────────────
print(f"\n✅ Exported {len(PIECE_LABELS)} pieces + board to {PIECE_OBJS_DIR}")
print(f"{'Label':<6} {'Faces':<8} {'Extents':<30} {'Color'}")
print('-' * 70)
for i, label in enumerate(PIECE_LABELS):
    p = pieces[i]
    color = PIECE_COLORS_LINEAR[label]
    print(f"{label:<6} {len(p.faces):<8} {str(np.round(p.extents, 1)):<30} {color}")

## Section 2 — HDRI Setup

In [ ]:
HDRI_DIR.mkdir(parents=True, exist_ok=True)

# Try downloading HDRIs from Poly Haven via BlenderProc
try:
    result = subprocess.run(
        ['blenderproc', 'download', 'haven', str(HDRI_DIR)],
        capture_output=True, text=True, timeout=300
    )
    print(result.stdout[-500:] if result.stdout else 'No stdout')
    if result.returncode != 0:
        print(f"Download returned code {result.returncode}")
except Exception as e:
    print(f"HDRI download failed: {e}")

# Check if any .hdr files exist
hdr_files = sorted(HDRI_DIR.glob('**/*.hdr'))

if not hdr_files:
    print("No .hdr files found. Generating procedural HDRIs...")
    
    def make_procedural_hdri(name, top_color, mid_color, bot_color):
        """Generate a simple gradient HDRI (equirectangular, 256×128, float32)."""
        h, w = 128, 256
        img = np.zeros((h, w, 3), dtype=np.float32)
        for y in range(h):
            t = y / (h - 1)  # 0 at top, 1 at bottom
            if t < 0.5:
                frac = t / 0.5
                color = np.array(top_color) * (1 - frac) + np.array(mid_color) * frac
            else:
                frac = (t - 0.5) / 0.5
                color = np.array(mid_color) * (1 - frac) + np.array(bot_color) * frac
            img[y, :] = color
        
        path = HDRI_DIR / f"{name}.hdr"
        cv2.imwrite(str(path), img)
        return path
    
    # Warm studio
    make_procedural_hdri('warm_studio',
        top_color=[1.2, 1.0, 0.8], mid_color=[0.8, 0.7, 0.6], bot_color=[0.4, 0.35, 0.3])
    
    # Cool daylight
    make_procedural_hdri('cool_daylight',
        top_color=[0.5, 0.7, 1.5], mid_color=[0.9, 0.95, 1.0], bot_color=[0.3, 0.35, 0.4])
    
    # Neutral diffuse
    make_procedural_hdri('neutral_diffuse',
        top_color=[0.9, 0.9, 0.9], mid_color=[0.7, 0.7, 0.7], bot_color=[0.5, 0.5, 0.5])
    
    hdr_files = sorted(HDRI_DIR.glob('**/*.hdr'))

print(f"\n✅ HDRI count: {len(hdr_files)}")
for f in hdr_files[:5]:
    print(f"   {f.name}")

## Section 3 — Placement Generator

In [ ]:
def generate_placements(num_pieces, board_extents_mm):
    """Generate random piece placements on the board.
    
    Args:
        num_pieces: number of pieces to place
        board_extents_mm: numpy array [x_extent, y_extent, z_extent] in mm
    
    Returns:
        list of placement dicts
    """
    board_half_x = board_extents_mm[0] / 2 * 0.001 * 0.85
    board_depth  = board_extents_mm[2] * 0.001 * 0.85
    
    placements = []
    for _ in range(num_pieces):
        label = random.choice(PIECE_LABELS)
        placements.append({
            'label':     label,
            'class_id':  LABEL_TO_CLASS_ID[label],
            'pos_x':     random.uniform(-board_half_x, board_half_x),
            'pos_z':     random.uniform(0.005, board_depth - 0.005),
            'rot_z_deg': random.uniform(0, 360),
            'flip_x':    random.random() < 0.5,
        })
    return placements

# Quick test
test_placements = generate_placements(3, BOARD_EXTENTS_MM)
for p in test_placements:
    print(f"  {p['label']} (class {p['class_id']}): pos=({p['pos_x']:.4f}, {p['pos_z']:.4f}), rot={p['rot_z_deg']:.1f}°, flip={p['flip_x']}")
print("\n✅ Placement generator ready")

## Section 4 — Dataset Generation Loop

In [ ]:
from collections import Counter

# ── Create dataset directories ────────────────────────────────────────────────
for split in ['train', 'val']:
    (DATASET_DIR / 'images' / split).mkdir(parents=True, exist_ok=True)
    (DATASET_DIR / 'labels' / split).mkdir(parents=True, exist_ok=True)
(DATASET_DIR / 'tmp').mkdir(parents=True, exist_ok=True)

# ── Build piece OBJ paths dict ────────────────────────────────────────────────
piece_obj_paths = {lbl: str(PIECE_OBJS_DIR / f"piece_{lbl}.obj") for lbl in PIECE_LABELS}

# ── Collect HDRI paths ────────────────────────────────────────────────────────
hdri_paths = sorted(HDRI_DIR.glob('**/*.hdr'))
if not hdri_paths:
    raise RuntimeError(f"No .hdr files found in {HDRI_DIR}! Run Section 2 first.")
print(f"Available HDRIs: {len(hdri_paths)}")

# ── Generation loop ───────────────────────────────────────────────────────────
stats = {}

for split, num_images in [('train', NUM_TRAIN_IMAGES), ('val', NUM_VAL_IMAGES)]:
    success_count = 0
    total_annotations = 0
    class_freq = Counter()
    
    print(f"\n{'='*60}")
    print(f"Generating {split} split: {num_images} images")
    print(f"{'='*60}")
    
    for img_idx in tqdm(range(num_images), desc=f"{split}"):
        # Random number of pieces
        n_pieces = random.randint(MIN_PIECES_PER_IMAGE, MAX_PIECES_PER_IMAGE)
        placements = generate_placements(n_pieces, BOARD_EXTENTS_MM)
        
        # Random HDRI
        hdri = str(random.choice(hdri_paths))
        
        # Output paths
        img_name = f"{split}_{img_idx:05d}.png"
        json_name = f"{split}_{img_idx:05d}.json"
        output_img = str(DATASET_DIR / 'images' / split / img_name)
        output_json = str(DATASET_DIR / 'tmp' / json_name)
        
        # Build args dict
        render_args = {
            'piece_obj_paths': piece_obj_paths,
            'board_obj_path':  str(PIECE_OBJS_DIR / 'board.obj'),
            'output_img_path': output_img,
            'output_json_path': output_json,
            'hdri_path':       hdri,
            'num_pieces':      n_pieces,
            'img_size':        IMG_SIZE,
            'cycles_samples':  CYCLES_SAMPLES,
            'use_gpu':         USE_GPU,
            'piece_placements': placements,
        }
        
        # Call BlenderProc
        cmd = ['blenderproc', 'run', str(GENERATOR_SCRIPT), '--', json.dumps(render_args)]
        
        try:
            result = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
            
            if result.returncode != 0:
                print(f"\n⚠️  Image {img_idx} failed (code {result.returncode})")
                if result.stderr:
                    print(f"   stderr: {result.stderr[-200:]}")
                # Write empty label file
                label_path = DATASET_DIR / 'labels' / split / f"{split}_{img_idx:05d}.txt"
                label_path.write_text('')
                continue
            
            # Read annotations from JSON
            if Path(output_json).exists():
                with open(output_json) as f:
                    ann_data = json.load(f)
                annotations = ann_data.get('annotations', [])
            else:
                annotations = []
            
            # Write YOLO label file
            label_path = DATASET_DIR / 'labels' / split / f"{split}_{img_idx:05d}.txt"
            with open(label_path, 'w') as f:
                for line in annotations:
                    f.write(line + '\n')
                    cls_id = int(line.split()[0])
                    class_freq[cls_id] += 1
            
            total_annotations += len(annotations)
            success_count += 1
            
        except subprocess.TimeoutExpired:
            print(f"\n⚠️  Image {img_idx} timed out")
            label_path = DATASET_DIR / 'labels' / split / f"{split}_{img_idx:05d}.txt"
            label_path.write_text('')
        except Exception as e:
            print(f"\n⚠️  Image {img_idx} error: {e}")
            label_path = DATASET_DIR / 'labels' / split / f"{split}_{img_idx:05d}.txt"
            label_path.write_text('')
    
    stats[split] = {
        'success': success_count,
        'total': num_images,
        'annotations': total_annotations,
        'class_freq': dict(class_freq),
    }

# ── Print summary ─────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"Generation Complete")
print(f"{'='*60}")
for split, s in stats.items():
    print(f"\n  {split}: {s['success']}/{s['total']} successful images")
    print(f"  Total annotation lines: {s['annotations']}")
    print(f"  Per-class frequency:")
    for cls_id in sorted(s['class_freq'].keys()):
        name = CLASS_NAMES.get(cls_id, f'?{cls_id}')
        count = s['class_freq'][cls_id]
        warning = ' ⚠️ LOW' if split == 'train' and count < 50 else ''
        print(f"    {cls_id:2d} ({name}): {count}{warning}")

## Section 5 — Dataset YAML

In [ ]:
# ── Write dataset.yaml ────────────────────────────────────────────────────────
yaml_content = f"""# IQ Noodles Synthetic Segmentation Dataset
# Generated by 01_generate_synthetic_data.ipynb

path: {DATASET_DIR.resolve()}
train: images/train
val: images/val

nc: 11

names:
"""
for cls_id, name in CLASS_NAMES.items():
    yaml_content += f"  {cls_id}: {name}\n"

yaml_path = DATASET_DIR / 'dataset.yaml'
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

# ── Write classes.txt ──────────────────────────────────────────────────────────
classes_path = DATASET_DIR / 'classes.txt'
with open(classes_path, 'w') as f:
    for cls_id in sorted(CLASS_NAMES.keys()):
        f.write(f"{CLASS_NAMES[cls_id]}\n")

print(f"✅ Dataset YAML: {yaml_path}")
print(f"✅ Classes file: {classes_path}")
print()
print(yaml_content)

## Section 6 — Validation & Preview

In [ ]:
from collections import Counter

# ── Validate image/label counts ───────────────────────────────────────────────
for split in ['train', 'val']:
    img_dir = DATASET_DIR / 'images' / split
    lbl_dir = DATASET_DIR / 'labels' / split
    imgs = sorted(img_dir.glob('*.png'))
    lbls = sorted(lbl_dir.glob('*.txt'))
    print(f"{split}: {len(imgs)} images, {len(lbls)} labels")
    assert len(imgs) == len(lbls), f"Mismatch in {split}: {len(imgs)} images vs {len(lbls)} labels"

# ── Validate label file contents ──────────────────────────────────────────────
train_labels = sorted((DATASET_DIR / 'labels' / 'train').glob('*.txt'))
sample_files = random.sample(train_labels, min(10, len(train_labels)))

errors = []
class_freq = Counter()

for lbl_file in sample_files:
    with open(lbl_file) as f:
        for line_no, line in enumerate(f, 1):
            parts = line.strip().split()
            if not parts:
                continue
            cls_id = int(parts[0])
            coords = parts[1:]
            
            if cls_id < 0 or cls_id > 10:
                errors.append(f"{lbl_file.name}:{line_no} class_id={cls_id} out of range")
            if len(coords) % 2 != 0:
                errors.append(f"{lbl_file.name}:{line_no} odd number of coords ({len(coords)})")
            if len(coords) < 8:
                errors.append(f"{lbl_file.name}:{line_no} too few coords ({len(coords)})")
            
            for c in coords:
                val = float(c)
                if val < 0 or val > 1:
                    errors.append(f"{lbl_file.name}:{line_no} coord {val} outside [0,1]")
                    break
            
            class_freq[cls_id] += 1

if errors:
    print(f"\n❌ {len(errors)} validation errors:")
    for e in errors[:10]:
        print(f"  {e}")
else:
    print(f"\n✅ All sampled labels valid!")

print(f"\nClass frequency (sampled):")
for cls_id in sorted(class_freq.keys()):
    print(f"  {cls_id:2d} ({CLASS_NAMES.get(cls_id, '?')}): {class_freq[cls_id]}")

In [ ]:
# ── Preview: draw polygons on sample images ───────────────────────────────────
PREVIEW_COLORS = [
    (82, 171, 16),    # A YellowGreen
    (238, 57, 79),    # B Red
    (32, 109, 217),   # C DarkBlue
    (199, 120, 185),  # D Purple
    (252, 105, 12),   # E Orange
    (8, 167, 232),    # F SkyBlue
    (31, 161, 91),    # G Green
    (182, 48, 72),    # H DarkRed
    (249, 214, 94),   # I Yellow
    (236, 113, 168),  # J Pink
    (133, 218, 187),  # K Teal
]

train_imgs = sorted((DATASET_DIR / 'images' / 'train').glob('*.png'))
sample_imgs = random.sample(train_imgs, min(6, len(train_imgs)))

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for ax_idx, img_path in enumerate(sample_imgs):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    
    # Load corresponding label
    lbl_path = DATASET_DIR / 'labels' / 'train' / (img_path.stem + '.txt')
    if lbl_path.exists():
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                cls_id = int(parts[0])
                coords = [float(x) for x in parts[1:]]
                points = []
                for j in range(0, len(coords), 2):
                    px = int(coords[j] * w)
                    py = int(coords[j+1] * h)
                    points.append([px, py])
                points = np.array(points, dtype=np.int32)
                color = PREVIEW_COLORS[cls_id % len(PREVIEW_COLORS)]
                cv2.polylines(img, [points], True, color, 2)
    
    axes[ax_idx].imshow(img)
    axes[ax_idx].set_title(img_path.name, fontsize=10)
    axes[ax_idx].axis('off')

# Hide unused axes
for i in range(len(sample_imgs), len(axes)):
    axes[i].axis('off')

plt.tight_layout()

# Save preview
preview_path = DATASET_DIR / 'preview.jpg'
plt.savefig(str(preview_path), dpi=100, bbox_inches='tight')
print(f"✅ Preview saved to {preview_path}")
plt.show()

## 02 — Phase 1: Train on Synthetic Data

In [ ]:
yaml_path = DATASET_DIR / 'dataset.yaml'
assert yaml_path.exists(), f"Synthetic dataset YAML not found at {yaml_path}. Run notebook 01 first!"

### 1. Load Model & Train Phase 1

In [ ]:
from ultralytics import YOLO

# ── Model selection ───────────────────────────────────────────────────────────
MODEL_SIZE = 'yolo26s-seg.pt'
model = YOLO(MODEL_SIZE)

print(f"✅ Loaded {MODEL_SIZE} model")
print(f"   Parameters: {sum(p.numel() for p in model.model.parameters()):,}")

In [ ]:
# Phase 1: Train on synthetic data
results_phase1 = model.train(
    data=str(yaml_path),
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    patience=20,
    save=True,
    save_period=25,
    project=str(TRAINING_DIR),
    name='phase1_synthetic',
    exist_ok=True,
    # Augmentation settings
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.3,
    degrees=15.0,
    translate=0.1,
    scale=0.4,
    fliplr=0.5,
    flipud=0.1,
    mosaic=0.8,
    mixup=0.1,
    copy_paste=0.2,
)

print("\n✅ Phase 1 training complete (synthetic data)!")

### 2. Validate Phase 1

In [ ]:
# Phase 1 validation on synthetic val set
metrics_p1 = model.val(
    data=str(yaml_path),
    imgsz=640,
    batch=16,
    device=0,
)

print("\n=== Phase 1 Validation Metrics (Synthetic) ===")
print(f"  Box  mAP@0.5:     {metrics_p1.box.map50:.4f}")
print(f"  Box  mAP@0.5:0.95: {metrics_p1.box.map:.4f}")
print(f"  Mask mAP@0.5:     {metrics_p1.seg.map50:.4f}")
print(f"  Mask mAP@0.5:0.95: {metrics_p1.seg.map:.4f}")

print("\n=== Per-Class mAP@0.5 (Mask) ===")
if hasattr(metrics_p1.seg, 'ap50') and metrics_p1.seg.ap50 is not None:
    for i, ap in enumerate(metrics_p1.seg.ap50):
        label = PIECE_LABELS[i] if i < len(PIECE_LABELS) else f'Class {i}'
        color = PIECE_COLORS.get(label, ('?', (0,0,0)))[0]
        print(f"  Piece {label} ({color}): {ap:.4f}")

## 03 — Prepare Real Data (Roboflow)

### 1. Download Real Data from Roboflow

In [ ]:
# !pip install -q roboflow

from roboflow import Roboflow

# ⚠️  FILL IN your Roboflow project details below
ROBOFLOW_API_KEY   = "rtBCMfDQl6zSFnAQzsHn"
ROBOFLOW_WORKSPACE = "abdulsalams-workspace-cslqv"
ROBOFLOW_PROJECT   = "noodels"
ROBOFLOW_VERSION   = 1

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)

# Download in YOLOv8 segmentation format
dataset = project.version(ROBOFLOW_VERSION).download("yolov8")

REAL_DATA_DIR = Path(dataset.location)
print(f"\n✅ Real dataset downloaded to: {REAL_DATA_DIR}")
print(f"   Train: {REAL_DATA_DIR / 'train'}")
print(f"   Val:   {REAL_DATA_DIR / 'valid'}")

### 2. Validate Roboflow Labels

In [ ]:
# ── Read Roboflow's data.yaml ─────────────────────────────────────────────────
rf_yaml_path = REAL_DATA_DIR / 'data.yaml'
with open(rf_yaml_path) as f:
    rf_config = yaml.safe_load(f)

rf_names = rf_config.get('names', {})
if isinstance(rf_names, list):
    rf_names = {i: n for i, n in enumerate(rf_names)}

print(f"Unified target mapping ({NUM_CLASSES} classes):")
for k, v in UNIFIED_NAMES.items():
    print(f"  {k}: {v}")

print("\nRoboflow class mapping:")
for k, v in sorted(rf_names.items(), key=lambda x: int(x[0])):
    print(f"  {k}: {v}")

# ── Verify match ──────────────────────────────────────────────────────────────
remap = {}
mismatches = []

for rf_id, rf_name in rf_names.items():
    rf_id = int(rf_id)
    if rf_id in UNIFIED_NAMES:
        uni_name = UNIFIED_NAMES[rf_id]
        rf_letter = rf_name.strip().split('_')[0].upper()
        uni_letter = uni_name.split('_')[0].upper()
        if rf_letter == uni_letter:
            remap[rf_id] = rf_id
        else:
            mismatches.append(f"  ID {rf_id}: Roboflow='{rf_name}' vs Unified='{uni_name}'")
            remap[rf_id] = rf_id
    else:
        mismatches.append(f"  ID {rf_id}: '{rf_name}' — not in unified scheme (>10)")

if mismatches:
    print(f"\n⚠️  {len(mismatches)} potential mismatches:")
    for m in mismatches:
        print(m)
    print("\nProceeding anyway — class IDs will be kept as-is.")
else:
    print(f"\n✅ All {len(remap)} Roboflow classes match unified scheme perfectly!")
    print("   No remapping needed — labels will be copied as-is.")

### 3. Prepare Fine-Tune Dataset

In [ ]:
FINETUNE_DIR.mkdir(parents=True, exist_ok=True)

stats = {'copied': 0, 'skipped_lines': 0, 'total_files': 0, 'total_images': 0}

for split_name, rf_split in [('train', 'train'), ('val', 'valid'), ('val', 'val')]:
    rf_img_dir = REAL_DATA_DIR / rf_split / 'images'
    rf_lbl_dir = REAL_DATA_DIR / rf_split / 'labels'

    if not rf_img_dir.exists() or not rf_lbl_dir.exists():
        continue

    out_img_dir = FINETUNE_DIR / 'images' / split_name
    out_lbl_dir = FINETUNE_DIR / 'labels' / split_name
    out_img_dir.mkdir(parents=True, exist_ok=True)
    out_lbl_dir.mkdir(parents=True, exist_ok=True)

    label_files = list(rf_lbl_dir.glob('*.txt'))
    print(f"\n{rf_split} → {split_name}: {len(label_files)} label files")

    for lbl_file in label_files:
        valid_lines = []
        with open(lbl_file) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 7:
                    stats['skipped_lines'] += 1
                    continue
                cls_id = int(parts[0])
                if 0 <= cls_id < NUM_CLASSES:
                    valid_lines.append(line.strip())
                    stats['copied'] += 1
                else:
                    stats['skipped_lines'] += 1

        out_lbl = out_lbl_dir / lbl_file.name
        with open(out_lbl, 'w') as f:
            f.write('\n'.join(valid_lines) + '\n' if valid_lines else '')
        stats['total_files'] += 1

        img_stem = lbl_file.stem
        for ext in ['.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG']:
            src_img = rf_img_dir / (img_stem + ext)
            if src_img.exists():
                dst_img = out_img_dir / src_img.name
                if not dst_img.exists():
                    shutil.copy2(src_img, dst_img)
                stats['total_images'] += 1
                break

print(f"\n{'='*50}")
print(f"✅ Dataset prepared!")
print(f"   Label files: {stats['total_files']}")
print(f"   Images:      {stats['total_images']}")
print(f"   Annotations: {stats['copied']}")
if stats['skipped_lines']:
    print(f"   ⚠️  Skipped:  {stats['skipped_lines']} invalid lines")

### 4. Create Fine-Tune Dataset YAML

In [ ]:
finetune_yaml_content = f"""# IQ Noodles Fine-Tune Dataset — Real Photos (Phase 2)
# 11 classes (A-K), IDs 0-10 — matches Phase 1 synthetic training

path: {FINETUNE_DIR.resolve()}
train: images/train
val: images/val

nc: {NUM_CLASSES}

names:
"""
for idx, name in UNIFIED_NAMES.items():
    finetune_yaml_content += f"  {idx}: {name}\n"

finetune_yaml_path = FINETUNE_DIR / 'dataset.yaml'
with open(finetune_yaml_path, 'w') as f:
    f.write(finetune_yaml_content)

print(f"✅ Fine-tune dataset YAML: {finetune_yaml_path}")
print(finetune_yaml_content)

### 5. Validate Labels

In [ ]:
def validate_labels(dataset_dir, split='train'):
    """Check label files for correctness."""
    lbl_dir = Path(dataset_dir) / 'labels' / split
    img_dir = Path(dataset_dir) / 'images' / split

    label_files = sorted(lbl_dir.glob('*.txt'))
    image_files = sorted(img_dir.glob('*'))
    image_stems = {f.stem for f in image_files}

    print(f"  Images: {len(image_files)}  |  Labels: {len(label_files)}")

    class_counts = Counter()
    orphan_labels = []
    bad_lines = []
    total_annotations = 0

    for lf in label_files:
        if lf.stem not in image_stems:
            orphan_labels.append(lf.name)

        with open(lf) as f:
            for line_no, line in enumerate(f, 1):
                parts = line.strip().split()
                if not parts:
                    continue
                cls_id = int(parts[0])
                n_coords = len(parts) - 1

                if cls_id < 0 or cls_id >= len(UNIFIED_NAMES):
                    bad_lines.append(f"{lf.name}:{line_no} class_id={cls_id} out of range")
                if n_coords < 6 or n_coords % 2 != 0:
                    bad_lines.append(f"{lf.name}:{line_no} invalid polygon ({n_coords} coords)")
                else:
                    coords = [float(x) for x in parts[1:]]
                    if any(c < 0 or c > 1 for c in coords):
                        bad_lines.append(f"{lf.name}:{line_no} coords outside [0,1]")

                class_counts[cls_id] += 1
                total_annotations += 1

    print(f"  Total annotations: {total_annotations}")
    print(f"  Class distribution:")
    for cls_id in sorted(class_counts.keys()):
        name = UNIFIED_NAMES.get(cls_id, f"UNKNOWN_{cls_id}")
        print(f"    {cls_id:2d} ({name}): {class_counts[cls_id]}")

    if orphan_labels:
        print(f"\n  ⚠️  {len(orphan_labels)} label files without matching image")
    if bad_lines:
        print(f"\n  ❌ {len(bad_lines)} problematic lines:")
        for bl in bad_lines[:10]:
            print(f"    {bl}")
    else:
        print(f"\n  ✅ All labels valid!")

    return len(bad_lines) == 0

print("=== Train split ===")
train_ok = validate_labels(FINETUNE_DIR, 'train')
print("\n=== Val split ===")
val_ok = validate_labels(FINETUNE_DIR, 'val')

if train_ok and val_ok:
    print("\n✅ All labels validated — ready for fine-tuning!")
else:
    print("\n⚠️  Fix label issues above before proceeding")

## 04 — Phase 2: Fine-Tune on Real Data

In [ ]:
p1_best = TRAINING_DIR / 'phase1_synthetic' / 'weights' / 'best.pt'
finetune_yaml = FINETUNE_DIR / 'dataset.yaml'

### 1. Inspect Available Datasets

In [ ]:
for dataset_name in ['noodles_seg_dataset', 'noodles_finetune_dataset']:
    base = DATA_DIR / dataset_name
    print(f"\n📁 {dataset_name}/")
    for split in ['train', 'val']:
        img_dir = base / 'images' / split
        lbl_dir = base / 'labels' / split
        n_imgs = len(list(img_dir.glob('*'))) if img_dir.exists() else 0
        n_lbls = len(list(lbl_dir.glob('*.txt'))) if lbl_dir.exists() else 0
        print(f"  {split}: {n_imgs} images, {n_lbls} labels")
    
    yaml_file = base / 'dataset.yaml'
    if yaml_file.exists():
        with open(yaml_file) as f:
            cfg = yaml.safe_load(f)
        print(f"  YAML: nc={cfg.get('nc')}, names={list(cfg.get('names', {}).values())[:3]}...")
    else:
        print(f"  YAML: ❌ not found")

### 2. Create Val Split from Train (if needed)

In [ ]:
train_imgs = FINETUNE_DIR / 'images' / 'train'
val_imgs   = FINETUNE_DIR / 'images' / 'val'
train_lbls = FINETUNE_DIR / 'labels' / 'train'
val_lbls   = FINETUNE_DIR / 'labels' / 'val'

n_val = len(list(val_imgs.glob('*'))) if val_imgs.exists() else 0
n_train = len(list(train_imgs.glob('*'))) if train_imgs.exists() else 0

VAL_FRACTION = 0.2

if n_val == 0 and n_train > 0:
    print(f"No val set found. Splitting {VAL_FRACTION*100:.0f}% from train ({n_train} images)...")
    
    val_imgs.mkdir(parents=True, exist_ok=True)
    val_lbls.mkdir(parents=True, exist_ok=True)
    
    all_imgs = sorted(train_imgs.glob('*'))
    n_val_target = max(1, int(len(all_imgs) * VAL_FRACTION))
    val_selection = random.sample(all_imgs, n_val_target)
    
    for img_file in val_selection:
        shutil.move(str(img_file), str(val_imgs / img_file.name))
        lbl_src = train_lbls / (img_file.stem + '.txt')
        if lbl_src.exists():
            shutil.move(str(lbl_src), str(val_lbls / lbl_src.name))
    
    print(f"  Moved {len(val_selection)} images+labels to val")
    print(f"  Train: {len(list(train_imgs.glob('*')))} images remaining")
    print(f"  Val:   {len(list(val_imgs.glob('*')))} images")
else:
    print(f"Val set exists with {n_val} images. No split needed.")

### 3. Fine-Tune (Phase 2)

In [ ]:
from ultralytics import YOLO

# ── Load Phase 1 best weights ─────────────────────────────────────────────────
p1_best = TRAINING_DIR / 'phase1_synthetic' / 'weights' / 'best.pt'
if not p1_best.exists():
    p1_best = TRAINING_DIR / 'phase1_synthetic' / 'weights' / 'last.pt'
assert p1_best.exists(), f"Phase 1 weights not found! Run notebook 02 first."

model = YOLO(str(p1_best))
print(f"✅ Loaded Phase 1 weights: {p1_best}")

# ── Fine-tune ─────────────────────────────────────────────────────────────────
results_phase2 = model.train(
    data=str(finetune_yaml),
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    patience=15,
    freeze=10,
    lr0=0.001,
    save=True,
    save_period=10,
    project=str(TRAINING_DIR),
    name='phase2_finetune',
    exist_ok=True,
    # Lighter augmentation for real data
    hsv_h=0.01,
    hsv_s=0.3,
    hsv_v=0.2,
    degrees=10.0,
    translate=0.1,
    scale=0.3,
    fliplr=0.5,
    flipud=0.0,
    mosaic=0.5,
    mixup=0.05,
    copy_paste=0.1,
)

print("\n✅ Phase 2 fine-tuning complete!")

### 4. Validate & Compare

In [ ]:
# Validate Phase 2 on finetune val set
metrics_p2 = model.val(
    data=str(finetune_yaml),
    imgsz=640,
    batch=16,
    device=0,
)

print("\n=== Phase 2 Validation Metrics (Real Data) ===")
print(f"  Box  mAP@0.5:     {metrics_p2.box.map50:.4f}")
print(f"  Box  mAP@0.5:0.95: {metrics_p2.box.map:.4f}")
print(f"  Mask mAP@0.5:     {metrics_p2.seg.map50:.4f}")
print(f"  Mask mAP@0.5:0.95: {metrics_p2.seg.map:.4f}")

# ── Compare with Phase 1 ──────────────────────────────────────────────────────
# Load Phase 1 results if available
p1_csv = TRAINING_DIR / 'phase1_synthetic' / 'results.csv'
if p1_csv.exists():
    import pandas as pd
    p1_results = pd.read_csv(p1_csv)
    # Get last epoch metrics
    last = p1_results.iloc[-1]
    # Column names may vary, look for map50 columns
    map50_cols = [c for c in p1_results.columns if 'map50' in c.lower() and 'map50-95' not in c.lower()]
    if map50_cols:
        p1_map50 = last[map50_cols[0]]
        print(f"\n=== Comparison ===")
        print(f"  Phase 1 mAP50: {p1_map50:.4f} (synthetic val)")
        print(f"  Phase 2 mAP50: {metrics_p2.seg.map50:.4f} (real val)")
else:
    print(f"\n⚠️  Phase 1 results.csv not found at {p1_csv}")

## 05 — Export & Evaluate

### 1. Find Best Model

In [ ]:
# Prefer Phase 2 best.pt → Phase 1 best.pt → Phase 2 last.pt → Phase 1 last.pt
candidates = [
    TRAINING_DIR / 'phase2_finetune'   / 'weights' / 'best.pt',
    TRAINING_DIR / 'phase1_synthetic'  / 'weights' / 'best.pt',
    TRAINING_DIR / 'phase2_finetune'   / 'weights' / 'last.pt',
    TRAINING_DIR / 'phase1_synthetic'  / 'weights' / 'last.pt',
]

best_model_path = None
for c in candidates:
    if c.exists():
        best_model_path = c
        break

if best_model_path:
    print(f"✅ Best model: {best_model_path}")
    print(f"   Size: {best_model_path.stat().st_size / 1024 / 1024:.1f} MB")
else:
    print("❌ No model weights found! Run notebooks 02/04 first.")

### 2. Export to TorchScript

In [ ]:
from ultralytics import YOLO

assert best_model_path and best_model_path.exists(), "No model to export!"

model = YOLO(str(best_model_path))
export_path = model.export(format='torchscript')

print(f"\n✅ Exported: {export_path}")
print(f"   Size: {Path(export_path).stat().st_size / 1024 / 1024:.1f} MB")

### 3. Package Model

In [ ]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Determine phase directory
p2_dir = TRAINING_DIR / 'phase2_finetune'
p1_dir = TRAINING_DIR / 'phase1_synthetic'
phase_dir = p2_dir if p2_dir.exists() else p1_dir

zip_path = MODELS_DIR / 'noodles_yolo_seg_trained.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    # Add best/last weights
    for wt in ['best.pt', 'last.pt', 'best.torchscript']:
        wt_path = phase_dir / 'weights' / wt
        if wt_path.exists():
            zf.write(wt_path, f"weights/{wt}")
            print(f"  + weights/{wt}")
    
    # Add results CSV and plots
    for fname in ['results.csv', 'results.png', 'confusion_matrix.png', 'labels.jpg']:
        fpath = phase_dir / fname
        if fpath.exists():
            zf.write(fpath, fname)
            print(f"  + {fname}")
    
    # Add Phase 1 weights too if Phase 2 exists
    if phase_dir == p2_dir and (p1_dir / 'weights' / 'best.pt').exists():
        p1_best = p1_dir / 'weights' / 'best.pt'
        zf.write(p1_best, 'phase1_weights/best.pt')
        print(f"  + phase1_weights/best.pt")
    
    # Add dataset configs
    for ds_yaml in [DATASET_DIR / 'dataset.yaml', FINETUNE_DIR / 'dataset.yaml']:
        if ds_yaml.exists():
            arcname = f"configs/{ds_yaml.parent.name}_{ds_yaml.name}"
            zf.write(ds_yaml, arcname)
            print(f"  + {arcname}")
    
    classes_file = DATASET_DIR / 'classes.txt'
    if classes_file.exists():
        zf.write(classes_file, 'classes.txt')
        print(f"  + classes.txt")

print(f"\n✅ Packaged model: {zip_path}")
print(f"   Size: {zip_path.stat().st_size / 1024 / 1024:.1f} MB")

### 4. Test on Local Photo (Optional)

In [ ]:
import matplotlib.pyplot as plt
import cv2
import numpy as np

# Set a test image path or None to skip
TEST_IMAGE = None  # e.g. str(PROJECT_ROOT / 'test_photo.jpg')

if TEST_IMAGE and Path(TEST_IMAGE).exists():
    model = YOLO(str(best_model_path))
    results = model.predict(source=TEST_IMAGE, imgsz=640, conf=0.3)
    
    # Draw results
    for r in results:
        annotated = r.plot()
        plt.figure(figsize=(10, 10))
        plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
        plt.title("Test Photo Prediction")
        plt.axis('off')
        plt.show()
        
        if r.boxes is not None:
            print(f"\nDetected {len(r.boxes)} pieces:")
            for box in r.boxes:
                cls_id = int(box.cls)
                conf = float(box.conf)
                label = PIECE_LABELS[cls_id] if cls_id < len(PIECE_LABELS) else '?'
                color = PIECE_COLORS.get(label, ('?', (0,0,0)))[0]
                print(f"  Piece {label} ({color}): conf={conf:.3f}")
else:
    print("No test image specified. Set TEST_IMAGE to a path to test.")

### 5. Summary

In [ ]:
print("="*60)
print("IQ Noodles — Training Pipeline Summary")
print("="*60)
print(f"\nProject root: {PROJECT_ROOT}")
print(f"Data dir:     {DATA_DIR}")

# Check what exists
items = [
    ('Synthetic dataset',  DATASET_DIR / 'dataset.yaml'),
    ('Real finetune data', FINETUNE_DIR / 'dataset.yaml'),
    ('Phase 1 weights',    TRAINING_DIR / 'phase1_synthetic' / 'weights' / 'best.pt'),
    ('Phase 2 weights',    TRAINING_DIR / 'phase2_finetune' / 'weights' / 'best.pt'),
    ('Exported model',     zip_path if 'zip_path' in dir() else Path('nonexistent')),
]

for label, path in items:
    status = '✅' if path.exists() else '❌'
    size = f"({path.stat().st_size / 1024 / 1024:.1f} MB)" if path.exists() and path.is_file() else ''
    print(f"  {status} {label}: {path.name} {size}")

print(f"\n{'='*60}")
print("Done! 🎉")